# Strategy Diversity Evaluation Framework: Analysis Notebook

This notebook contains the main paper-facing analyses and figure-generation cells for the strategy diversity evaluation framework. It is a cleaned public version with scratch cells, deprecated R-level analysis, and saved outputs removed.

## Distinct Strategy Analysis

This section compares the number of **valid and correct distinct strategies** produced by each model on the 80 benchmark problems.

Important note on the AoPS reference:
- `s1, s2, ...` are AoPS/reference strategies from `complete_strategy_families_AoPS_LLM.csv`.
- `n1, n2, ...` are novel strategies generated by AI models.
- For the AoPS baseline below, we count only the reference strategies with IDs matching `s\d+`.

For models, a strategy counts only if it is both:
- `final_strategy_valid = 1`
- `final_result_correct = 1`

and has a non-empty `final_distinct_strategy_id`.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path('.')
dataset_root = repo_root.parent / 'dataset_release'
strategy_file = dataset_root / 'full80_prompt_multi_annotation' / 'Full80_valid_correct_strategies_annotated.csv'
reference_file = dataset_root / 'strategy_inventory' / 'complete_strategy_families_AoPS_LLM.csv'
analysis_dir = repo_root / 'analysis_outputs'
analysis_dir.mkdir(exist_ok=True)

model_df = pd.read_csv(strategy_file)
reference_df = pd.read_csv(reference_file)

problem_meta = (
    reference_df[['Problem_ID', 'Problem', 'Domain']]
    .drop_duplicates(subset=['Problem_ID'])
    .rename(columns={'Problem_ID': 'problem_id', 'Problem': 'problem', 'Domain': 'domain'})
)
problem_meta['problem_id'] = problem_meta['problem_id'].astype(str).str.strip()

model_df['problem_id'] = model_df['problem_id'].astype(str).str.strip()
model_df['benchmark_model'] = model_df['generated_model_id'].astype(str).str.strip()
model_df = model_df.merge(problem_meta[['problem_id', 'domain']], on='problem_id', how='left', validate='m:1')

countable = model_df[model_df['count_as_distinct_strategy'].fillna(0).astype(int).eq(1)].copy()
countable['final_distinct_strategy_id'] = countable['final_distinct_strategy_id'].astype(str).str.strip()
countable = countable[countable['final_distinct_strategy_id'].ne('')]

models = sorted(model_df['benchmark_model'].dropna().unique())
all_problem_model_pairs = (
    problem_meta[['problem_id', 'domain']]
    .assign(_k=1)
    .merge(pd.DataFrame({'benchmark_model': models, '_k': 1}), on='_k')
    .drop(columns=['_k'])
)

model_problem_counts = (
    countable.groupby(['problem_id', 'domain', 'benchmark_model'])['final_distinct_strategy_id']
    .nunique()
    .reset_index(name='distinct_strategy_count')
)
model_problem_counts = all_problem_model_pairs.merge(
    model_problem_counts, on=['problem_id', 'domain', 'benchmark_model'], how='left'
)
model_problem_counts['distinct_strategy_count'] = model_problem_counts['distinct_strategy_count'].fillna(0).astype(int)

reference_df['Problem_ID'] = reference_df['Problem_ID'].astype(str).str.strip()
reference_df['Strategy_ID'] = reference_df['Strategy_ID'].astype(str).str.strip()
reference_only = reference_df[reference_df['Strategy_ID'].str.fullmatch(r's\d+', na=False)].copy()

aops_problem_counts = (
    reference_only.groupby(['Problem_ID', 'Domain'])['Strategy_ID']
    .nunique()
    .reset_index()
    .rename(columns={'Problem_ID': 'problem_id', 'Domain': 'domain', 'Strategy_ID': 'distinct_strategy_count'})
)
aops_problem_counts['benchmark_model'] = 'AoPS'
aops_problem_counts = problem_meta[['problem_id', 'domain']].merge(
    aops_problem_counts, on=['problem_id', 'domain'], how='left'
)
aops_problem_counts['distinct_strategy_count'] = aops_problem_counts['distinct_strategy_count'].fillna(0).astype(int)

problem_counts_all = pd.concat([model_problem_counts, aops_problem_counts], ignore_index=True)

overall_compare = (
    problem_counts_all.groupby('benchmark_model')['distinct_strategy_count']
    .agg(total_distinct_strategies='sum', average_distinct_strategies='mean', median_distinct_strategies='median')
    .reset_index()
)
overall_compare['num_problems'] = 80
overall_compare = overall_compare[['benchmark_model', 'num_problems', 'total_distinct_strategies', 'average_distinct_strategies', 'median_distinct_strategies']]

domain_compare = (
    problem_counts_all.groupby(['domain', 'benchmark_model'])['distinct_strategy_count']
    .agg(total_distinct_strategies='sum', average_distinct_strategies='mean', median_distinct_strategies='median')
    .reset_index()
)
domain_compare['num_problems_in_domain'] = problem_counts_all.groupby(['domain', 'benchmark_model'])['problem_id'].nunique().values
domain_compare = domain_compare[['domain', 'benchmark_model', 'num_problems_in_domain', 'total_distinct_strategies', 'average_distinct_strategies', 'median_distinct_strategies']]

overall_compare.to_csv(analysis_dir / 'distinct_strategy_counts_overall_by_model_with_aops.csv', index=False)
domain_compare.to_csv(analysis_dir / 'distinct_strategy_counts_by_domain_model_with_aops.csv', index=False)

overall_compare


In [ ]:
# Overall comparison plot: total and average number of distinct strategies
plot_order = ['AoPS', 'gpt-5.4', 'gemini-3.1-pro-preview', 'deepseek-reasoner', 'claude-opus-4-6']
label_map = {
    'AoPS': 'AoPS',
    'gpt-5.4': 'GPT',
    'gemini-3.1-pro-preview': 'Gemini',
    'deepseek-reasoner': 'DeepSeek',
    'claude-opus-4-6': 'Claude',
}
color_map = {
    'AoPS': '#4c566a',
    'gpt-5.4': '#4E79A7',
    'gemini-3.1-pro-preview': '#59A14F',
    'deepseek-reasoner': '#F28E2B',
    'claude-opus-4-6': '#E15759',
}

overall_plot = overall_compare.copy()
overall_plot['plot_order'] = overall_plot['benchmark_model'].map({m: i for i, m in enumerate(plot_order)})
overall_plot = overall_plot.sort_values('plot_order')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(
    overall_plot['benchmark_model'].map(label_map),
    overall_plot['total_distinct_strategies'],
    color=[color_map[m] for m in overall_plot['benchmark_model']]
)
axes[0].set_title('Total Distinct Strategies Across 80 Problems')
axes[0].set_ylabel('Total count')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(
    overall_plot['benchmark_model'].map(label_map),
    overall_plot['average_distinct_strategies'],
    color=[color_map[m] for m in overall_plot['benchmark_model']]
)
axes[1].set_title('Average Distinct Strategies per Problem')
axes[1].set_ylabel('Average count')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

### Radar Graph

Run the next cell to render the radar chart for average distinct strategies by domain.


In [ ]:
# Domain-level radar chart
import numpy as np
import matplotlib.pyplot as plt

domain_order = ['Algebra', 'Combinatorics', 'Geometry', 'Number Theory', 'Probability']
plot_order = ['AoPS', 'gpt-5.4', 'gemini-3.1-pro-preview', 'deepseek-reasoner', 'claude-opus-4-6']
label_map = {
    'AoPS': 'AoPS',
    'gpt-5.4': 'GPT',
    'gemini-3.1-pro-preview': 'Gemini',
    'deepseek-reasoner': 'DeepSeek',
    'claude-opus-4-6': 'Claude',
}
color_map = {
    'AoPS': '#4c566a',
    'gpt-5.4': '#4E79A7',
    'gemini-3.1-pro-preview': '#59A14F',
    'deepseek-reasoner': '#F28E2B',
    'claude-opus-4-6': '#E15759',
}

rdf = domain_compare.copy()
rdf['domain'] = pd.Categorical(rdf['domain'], categories=domain_order, ordered=True)
pivot_domain = rdf.pivot(index='domain', columns='benchmark_model', values='average_distinct_strategies')[plot_order]
pivot_domain = pivot_domain.reindex(domain_order)

labels = domain_order
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for model in plot_order:
    values = pivot_domain[model].tolist()
    values += values[:1]
    ax.plot(angles, values, label=label_map[model], color=color_map[model], linewidth=2.4)
    if model == 'AoPS':
        ax.fill(angles, values, color=color_map[model], alpha=0.06)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=12)
ax.set_title('Average Distinct Strategies per Problem by Domain', pad=20)
ax.set_rlabel_position(20)
ax.tick_params(axis='y', labelsize=11)
ax.grid(alpha=0.4)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.16), ncol=3, title='Source / Model', frameon=False)
plt.tight_layout()
plt.show()

## AoPS vs Novel Strategies

This section separates **valid and correct distinct strategies** into two sources:
- `AoPS_reference`: strategy IDs like `s1, s2, ...`
- `Novel_AI`: strategy IDs like `n1, n2, ...`

These counts are based on the finalized file `Full80_valid_correct_strategies_annotated.csv`.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path('.')
dataset_root = repo_root.parent / 'dataset_release'
strategy_file = dataset_root / 'full80_prompt_multi_annotation' / 'Full80_valid_correct_strategies_annotated.csv'
analysis_dir = repo_root / 'analysis_outputs'
analysis_dir.mkdir(exist_ok=True)

model_df = pd.read_csv(strategy_file)
countable = model_df[model_df['count_as_distinct_strategy'].fillna(0).astype(int).eq(1)].copy()
countable['final_distinct_strategy_id'] = countable['final_distinct_strategy_id'].astype(str).str.strip()
countable = countable[countable['final_distinct_strategy_id'].ne('')]
countable['benchmark_model'] = countable['generated_model_id'].astype(str).str.strip()

def strategy_source_type(s):
    s = str(s).strip().lower()
    if s.startswith('s'):
        return 'AoPS_reference'
    if s.startswith('n'):
        return 'Novel_AI'
    return 'Other'

countable['strategy_source_type'] = countable['final_distinct_strategy_id'].map(strategy_source_type)

aops_novel_overall = (
    countable.groupby(['benchmark_model', 'strategy_source_type'])['final_distinct_strategy_id']
    .count()
    .reset_index(name='count')
)
aops_novel_overall['total_count'] = aops_novel_overall.groupby('benchmark_model')['count'].transform('sum')
aops_novel_overall['proportion'] = aops_novel_overall['count'] / aops_novel_overall['total_count']
aops_novel_overall.to_csv(analysis_dir / 'aops_vs_novel_overall_by_model.csv', index=False)

aops_novel_overall


In [ ]:
# Overall AoPS-reference vs novel proportions
label_map = {
    'gpt-5.4': 'GPT',
    'gemini-3.1-pro-preview': 'Gemini',
    'deepseek-reasoner': 'DeepSeek',
    'claude-opus-4-6': 'Claude',
}
source_colors = {
    'AoPS_reference': '#4c78a8',
    'Novel_AI': '#f58518',
}

plot_df = aops_novel_overall.copy()
plot_df['benchmark_model_label'] = plot_df['benchmark_model'].map(label_map)
pivot_df = plot_df.pivot(index='benchmark_model_label', columns='strategy_source_type', values='proportion').fillna(0)
pivot_df = pivot_df.reindex(['GPT', 'Gemini', 'DeepSeek', 'Claude'])

ax = pivot_df.plot(
    kind='bar',
    stacked=True,
    figsize=(8, 5),
    color=[source_colors[c] for c in pivot_df.columns],
    width=0.75
)
ax.set_title('Proportion of AoPS-reference vs Novel Distinct Strategies')
ax.set_ylabel('Proportion')
ax.set_xlabel('Model')
ax.legend(title='Strategy source', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()


## Paired Gap to AoPS

This section compares each model to AoPS at the **problem level** using paired differences in the number of finalized valid-and-correct distinct strategies.

For each problem and model, we compute:
- `gap = model_distinct_strategies - AoPS_distinct_strategies`
- 95% bootstrap confidence intervals for the mean paired gap
- paired sign-flip permutation p-values


In [ ]:
overall_gap = pd.read_csv(analysis_dir / 'aops_model_paired_gap_overall.csv')
domain_gap = pd.read_csv(analysis_dir / 'aops_model_paired_gap_by_domain.csv')
rgroup_gap = pd.read_csv(analysis_dir / 'aops_model_paired_gap_by_rlevel_group.csv')
model_order = ['GPT', 'Gemini', 'DeepSeek', 'Claude']
colors = {'GPT':'#4E79A7','Gemini':'#59A14F','DeepSeek':'#F28E2B','Claude':'#E15759'}
overall_gap

In [ ]:
# Domain-gap heatmap for the main-text figure
# Tweak the sizing parameters below as needed.
import numpy as np

domain_gap = pd.read_csv(analysis_dir / 'aops_model_paired_gap_by_domain.csv')
row_order = ['Algebra', 'Combinatorics', 'Geometry', 'Number Theory', 'Probability']
col_order = ['GPT', 'Gemini', 'DeepSeek', 'Claude']

heat = (
    domain_gap[['group_value', 'benchmark_model_label', 'mean_gap_model_minus_aops']]
      .rename(columns={'group_value': 'Domain', 'benchmark_model_label': 'Model', 'mean_gap_model_minus_aops': 'Gap'})
      .pivot(index='Domain', columns='Model', values='Gap')
      .loc[row_order, col_order]
)

figsize = (11.2, 7.2)
xlab_fs = 20
ylab_fs = 20
tick_fs = 18
annot_fs = 17
cbar_tick_fs = 16
cbar_label_fs = 17

fig, ax = plt.subplots(figsize=figsize)
im = ax.imshow(heat.values, cmap='RdBu_r', vmin=-2.3, vmax=0.2, aspect='auto')

ax.set_xticks(np.arange(len(col_order)))
ax.set_yticks(np.arange(len(row_order)))
ax.set_xticklabels(col_order, fontsize=tick_fs)
ax.set_yticklabels(row_order, fontsize=tick_fs)
ax.set_xlabel('Model', fontsize=xlab_fs, labelpad=12)
ax.set_ylabel('Domain', fontsize=ylab_fs, labelpad=12)
ax.tick_params(axis='both', length=0)

for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        value = heat.iloc[i, j]
        ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=annot_fs, color='black')

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xticks(np.arange(-.5, len(col_order), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(row_order), 1), minor=True)
ax.grid(which='minor', color='white', linestyle='-', linewidth=1.1)
ax.tick_params(which='minor', bottom=False, left=False)

cbar = fig.colorbar(im, ax=ax, shrink=0.98, pad=0.02)
cbar.ax.tick_params(labelsize=cbar_tick_fs)
cbar.set_label('Mean paired gap vs AoPS', size=cbar_label_fs)

plt.tight_layout(pad=0.6)
plt.savefig(paper_fig_dir / 'figure7_domain_gap_heatmap_preview.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
plot = overall_gap.copy()
plot['benchmark_model_label'] = pd.Categorical(plot['benchmark_model_label'], categories=model_order, ordered=True)
plot = plot.sort_values('benchmark_model_label')
fig, ax = plt.subplots(figsize=(7,4.5))
ax.axhline(0, color='black', linewidth=1, linestyle='--')
for i, row in enumerate(plot.itertuples(index=False)):
    ax.errorbar(i, row.mean_gap_model_minus_aops, yerr=[[row.mean_gap_model_minus_aops - row.gap_bootstrap_ci_lower], [row.gap_bootstrap_ci_upper - row.mean_gap_model_minus_aops]], fmt='o', color=colors[row.benchmark_model_label], capsize=4, markersize=8)
ax.set_xticks(range(len(plot)))
ax.set_xticklabels(plot['benchmark_model_label'])
ax.set_ylabel('Mean paired gap vs AoPS
(model distinct strategies - AoPS)')
ax.set_title('Overall Strategy Diversity Gap Relative to AoPS')
plt.tight_layout()
plt.show()

In [ ]:
plot = domain_gap.copy()
domain_order = ['Algebra','Combinatorics','Geometry','Number Theory','Probability']
plot['group_value'] = pd.Categorical(plot['group_value'], categories=domain_order, ordered=True)
plot['benchmark_model_label'] = pd.Categorical(plot['benchmark_model_label'], categories=model_order, ordered=True)
plot = plot.sort_values(['group_value','benchmark_model_label'])
fig, ax = plt.subplots(figsize=(9,5.5))
ax.axvline(0, color='black', linewidth=1, linestyle='--')
y_positions = []
y_labels = []
for d_i, domain_name in enumerate(domain_order):
    sub = plot[plot['group_value'] == domain_name]
    for m_i, row in enumerate(sub.itertuples(index=False)):
        y = d_i*5 + m_i
        y_positions.append(y)
        y_labels.append(f'{domain_name} - {row.benchmark_model_label}')
        ax.errorbar(row.mean_gap_model_minus_aops, y, xerr=[[row.mean_gap_model_minus_aops - row.gap_bootstrap_ci_lower], [row.gap_bootstrap_ci_upper - row.mean_gap_model_minus_aops]], fmt='o', color=colors[row.benchmark_model_label], capsize=3, markersize=6)
ax.set_yticks(y_positions)
ax.set_yticklabels(y_labels, fontsize=9)
ax.set_xlabel('Mean paired gap vs AoPS')
ax.set_title('Domain-Level Strategy Diversity Gap Relative to AoPS')
plt.tight_layout()
plt.show()

## Overall Strategy Diversity Summary

This section gives a compact AoPS + 4 models summary for the main text. Distinct strategies here are finalized, valid, and correct.


In [ ]:
overall_diversity = pd.read_csv(analysis_dir / 'overall_strategy_diversity_summary_table.csv')
overall_diversity


In [ ]:
plot = overall_diversity.copy()
colors = {'AoPS':'#4c566a','GPT':'#4E79A7','Gemini':'#59A14F','DeepSeek':'#F28E2B','Claude':'#E15759'}
fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.bar(plot['source'], plot['avg_distinct_strategies_per_problem'], color=[colors[s] for s in plot['source']], width=0.72)
ax.set_title('Average Distinct Valid-Correct Strategies per Problem')
ax.set_xlabel('Source / Model')
ax.set_ylabel('Average distinct strategies')
ax.tick_params(axis='x', rotation=0)
for i, row in plot.iterrows():
    ax.text(i, row['avg_distinct_strategies_per_problem'] + 0.04, f"{row['avg_distinct_strategies_per_problem']:.2f}", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## Novel Strategy Overlap

This section examines whether the 50 single-run novel strategies are mostly model-specific or shared across models, and how those overlap patterns distribute across domains.

In [ ]:
share_counts = pd.read_csv(analysis_dir / 'novel_strategy_model_overlap_num_models.csv')
share_by_domain = pd.read_csv(analysis_dir / 'novel_strategy_model_overlap_num_models_by_domain.csv')
pattern_counts = pd.read_csv(analysis_dir / 'novel_strategy_model_overlap_patterns.csv')

share_counts

In [ ]:
# Main-text figure: how many novel strategies are unique vs shared by 2/3/4 models
plot = share_counts.sort_values('num_models').copy()
fig, ax = plt.subplots(figsize=(6.5, 4.5))
x = np.arange(len(plot))
ax.bar(x, plot['novel_strategy_count'], color=['#c7dcef','#91bde3','#5e9ccf','#2f6fa8'], edgecolor='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(plot['num_models'].astype(str))
ax.set_xlabel('Number of Models Sharing the Novel Strategy')
ax.set_ylabel('Number of Novel Strategies')
ax.set_title('Sharing of Novel Strategies Across Models')
for xi, yi in zip(x, plot['novel_strategy_count']):
    ax.text(xi, yi + 0.5, str(int(yi)), ha='center', va='bottom', fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Domain-level descriptive figure: unique-only novelty is split by model, while shared novelty is grouped
patterns_by_domain = pd.read_csv(analysis_dir / 'novel_strategy_model_overlap_patterns_by_domain.csv')
share_by_domain = pd.read_csv(analysis_dir / 'novel_strategy_model_overlap_num_models_by_domain.csv')

unique_only = patterns_by_domain[patterns_by_domain['num_models'] == 1].copy()
unique_only = unique_only.rename(columns={'pattern':'model', 'novel_strategy_count':'count'})
shared = share_by_domain[share_by_domain['num_models'].isin([2,3,4])].copy()
shared = shared.rename(columns={'novel_strategy_count':'count'})
shared['segment'] = shared['num_models'].map({2:'2 models',3:'3 models',4:'4 models'})

rows = []
for _, r in unique_only.iterrows():
    rows.append({'domain': r['domain'], 'segment': r['model'], 'count': r['count']})
for _, r in shared.iterrows():
    rows.append({'domain': r['domain'], 'segment': r['segment'], 'count': r['count']})
plot = pd.DataFrame(rows)

domain_order = ['Algebra','Combinatorics','Geometry','Number Theory','Probability']
segment_order = ['GPT','Gemini','DeepSeek','Claude','2 models','3 models','4 models']
pivot = plot.pivot(index='domain', columns='segment', values='count').fillna(0)
for c in segment_order:
    if c not in pivot.columns:
        pivot[c] = 0
pivot = pivot.loc[domain_order, segment_order]

colors = {
    'GPT':'#4c78a8',
    'Gemini':'#54a24b',
    'DeepSeek':'#e45756',
    'Claude':'#b279a2',
    '2 models':'#9ecae1',
    '3 models':'#5b8fc1',
    '4 models':'#244f7a',
}

fig, ax = plt.subplots(figsize=(8.5, 5.2))
x = np.arange(len(pivot))
bottom = np.zeros(len(pivot))
for seg in segment_order:
    vals = pivot[seg].to_numpy()
    ax.bar(x, vals, bottom=bottom, label=seg, color=colors[seg], edgecolor='black', linewidth=0.6)
    if seg in ['GPT','Gemini','DeepSeek','Claude']:
        for i, v in enumerate(vals):
            if v > 0:
                ax.text(x[i], bottom[i] + v/2, seg, ha='center', va='center', fontsize=8, rotation=90, color='white' if seg in ['GPT','DeepSeek','Claude'] else 'black')
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(pivot.index, rotation=20, ha='right')
ax.set_ylabel('Number of Novel Strategies')
ax.set_xlabel('Domain')
ax.set_title('Novel Strategy Sharing by Domain')
ax.legend(frameon=False, ncol=3, title='Overlap segment', bbox_to_anchor=(1.02,1), loc='upper left')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Appendix/notebook figure: an UpSet-like overlap pattern plot
patterns = pattern_counts.sort_values(['num_models','novel_strategy_count','pattern'], ascending=[False,False,True]).reset_index(drop=True).copy()
model_order = ['GPT','Gemini','DeepSeek','Claude']
fig = plt.figure(figsize=(10.5, 5.5))
gs = fig.add_gridspec(2, 1, height_ratios=[3.2, 1.7], hspace=0.08)
ax_top = fig.add_subplot(gs[0])
ax_bot = fig.add_subplot(gs[1], sharex=ax_top)
x = np.arange(len(patterns))
ax_top.bar(x, patterns['novel_strategy_count'], color='#4f81bd', edgecolor='black', linewidth=0.7)
for xi, yi in zip(x, patterns['novel_strategy_count']):
    ax_top.text(xi, yi + 0.25, str(int(yi)), ha='center', va='bottom', fontsize=9)
ax_top.set_ylabel('Novel strategies')
ax_top.set_title('Overlap Patterns of Novel Strategies Across Models')
ax_top.spines['top'].set_visible(False)
ax_top.spines['right'].set_visible(False)
ax_top.tick_params(axis='x', labelbottom=False)
for i, model in enumerate(model_order):
    y = len(model_order)-1-i
    ax_bot.hlines(y, -0.5, len(patterns)-0.5, color='#dddddd', linewidth=0.8)
    for j, pat in enumerate(patterns['pattern']):
        present = model in pat.split('+')
        ax_bot.scatter(j, y, s=80 if present else 28, color='#1f1f1f' if present else '#d0d0d0', zorder=3)
    ax_bot.text(-1.0, y, model, ha='right', va='center', fontsize=10)
ax_bot.set_yticks([])
ax_bot.set_xticks(x)
ax_bot.set_xticklabels(patterns['pattern'], rotation=45, ha='right', fontsize=9)
ax_bot.spines['top'].set_visible(False)
ax_bot.spines['right'].set_visible(False)
ax_bot.spines['left'].set_visible(False)
ax_bot.set_xlabel('Model overlap pattern')
plt.tight_layout()
plt.show()

In [ ]:
# Cleaner two-panel domain view: unique-only novelty by model, shared novelty by overlap size
patterns_by_domain = pd.read_csv(analysis_dir / 'novel_strategy_model_overlap_patterns_by_domain.csv')
share_by_domain = pd.read_csv(analysis_dir / 'novel_strategy_model_overlap_num_models_by_domain.csv')

domain_order = ['Algebra','Combinatorics','Geometry','Number Theory','Probability']
model_order = ['GPT','Gemini','DeepSeek','Claude']

unique_only = patterns_by_domain[patterns_by_domain['num_models'] == 1].copy()
unique_only = unique_only.rename(columns={'pattern':'model', 'novel_strategy_count':'count'})
unique_pivot = unique_only.pivot(index='domain', columns='model', values='count').fillna(0)
for m in model_order:
    if m not in unique_pivot.columns:
        unique_pivot[m] = 0
unique_pivot = unique_pivot.reindex(domain_order, fill_value=0)[model_order]

shared = share_by_domain[share_by_domain['num_models'].isin([2,3,4])].copy()
shared_pivot = shared.pivot(index='domain', columns='num_models', values='novel_strategy_count').fillna(0)
for c in [2,3,4]:
    if c not in shared_pivot.columns:
        shared_pivot[c] = 0
shared_pivot = shared_pivot.reindex(domain_order, fill_value=0)[[2,3,4]]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={'width_ratios':[1.35, 1]})
ax1, ax2 = axes
colors_models = {'GPT':'#4c78a8','Gemini':'#54a24b','DeepSeek':'#e45756','Claude':'#b279a2'}
colors_shared = {2:'#9ecae1',3:'#5b8fc1',4:'#244f7a'}
x = np.arange(len(domain_order))
width = 0.18
for i, model in enumerate(model_order):
    vals = unique_pivot[model].to_numpy()
    xpos = x + (i - 1.5) * width
    ax1.bar(xpos, vals, width=width, label=model, color=colors_models[model], edgecolor='black', linewidth=0.6)
    for xx, yy in zip(xpos, vals):
        if yy > 0:
            ax1.text(xx, yy + 0.08, str(int(yy)), ha='center', va='bottom', fontsize=8)
ax1.set_xticks(x)
ax1.set_xticklabels(domain_order, rotation=20, ha='right')
ax1.set_ylabel('Unique Novel Strategies')
ax1.set_title('A. Model-Specific Novel Strategies by Domain')
ax1.legend(frameon=False, ncol=2)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

bottom = np.zeros(len(domain_order))
for c in [2,3,4]:
    vals = shared_pivot[c].to_numpy()
    ax2.bar(x, vals, bottom=bottom, label=f'{c} models', color=colors_shared[c], edgecolor='black', linewidth=0.6)
    for xx, basey, yy in zip(x, bottom, vals):
        if yy > 0:
            ax2.text(xx, basey + yy/2, str(int(yy)), ha='center', va='center', fontsize=8, color='white' if c >= 3 else 'black')
    bottom += vals
ax2.set_xticks(x)
ax2.set_xticklabels(domain_order, rotation=20, ha='right')
ax2.set_ylabel('Shared Novel Strategies')
ax2.set_title('B. Shared Novel Strategies by Domain')
ax2.legend(frameon=False)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Fractional contribution to novel strategies by domain
# Each unique novel strategy is given total weight 1; if k models produced it, each gets 1/k credit.
novel_incidence = pd.read_csv(analysis_dir / 'novel_strategy_model_incidence.csv')
model_order = ['GPT','Gemini','DeepSeek','Claude']
domain_order = ['Algebra','Combinatorics','Geometry','Number Theory','Probability']

rows = []
for _, row in novel_incidence.iterrows():
    present_models = [m for m in model_order if int(row.get(m, 0)) == 1]
    if not present_models:
        continue
    w = 1.0 / len(present_models)
    for m in present_models:
        rows.append({'domain': row['domain'], 'model': m, 'credit': w})
credit_df = pd.DataFrame(rows)
summary = credit_df.groupby(['domain','model'], as_index=False)['credit'].sum()
domain_totals = summary.groupby('domain', as_index=False)['credit'].sum().rename(columns={'credit':'domain_total'})
summary = summary.merge(domain_totals, on='domain', how='left')
summary['contribution_pct'] = summary['credit'] / summary['domain_total']
pivot_pct = summary.pivot(index='domain', columns='model', values='contribution_pct').fillna(0)
for m in model_order:
    if m not in pivot_pct.columns:
        pivot_pct[m] = 0
pivot_pct = pivot_pct.reindex(domain_order, fill_value=0)[model_order]

colors = {'GPT':'#4E79A7','Gemini':'#59A14F','DeepSeek':'#F28E2B','Claude':'#E15759'}
fig, ax = plt.subplots(figsize=(8.2, 4.8))
x = np.arange(len(pivot_pct))
bottom = np.zeros(len(pivot_pct))
for m in model_order:
    vals = pivot_pct[m].to_numpy()
    ax.bar(x, vals, bottom=bottom, color=colors[m], edgecolor='black', linewidth=0.6, label=m)
    for i, (b, v) in enumerate(zip(bottom, vals)):
        if v >= 0.11:
            ax.text(x[i], b + v/2, f'{v*100:.0f}%', ha='center', va='center', fontsize=8, color='white' if m in ['GPT','DeepSeek','Claude'] else 'black')
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels(pivot_pct.index, rotation=20, ha='right')
ax.set_ylabel('Share of Novel Strategies in Domain')
ax.set_xlabel('Domain')
ax.set_title('Fractional Contribution to Novel Strategies by Domain')
ax.set_ylim(0,1)
ax.set_yticks(np.linspace(0,1,6))
ax.set_yticklabels([f'{int(t*100)}%' for t in np.linspace(0,1,6)])
ax.legend(frameon=False, ncol=2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Hybrid view: bar height = total novel strategies in each domain; stacked colors = model fractional contribution
summary = pd.read_csv(analysis_dir / 'novel_strategy_fractional_contribution_by_domain.csv')
model_order = ['GPT','Gemini','DeepSeek','Claude']
domain_order = ['Algebra','Combinatorics','Geometry','Number Theory','Probability']
colors = {'GPT':'#4E79A7','Gemini':'#59A14F','DeepSeek':'#F28E2B','Claude':'#E15759'}

pivot_credit = summary.pivot(index='domain', columns='model', values='credit').fillna(0)
for m in model_order:
    if m not in pivot_credit.columns:
        pivot_credit[m] = 0
pivot_credit = pivot_credit.reindex(domain_order, fill_value=0)[model_order]
domain_totals = pivot_credit.sum(axis=1)
pct = pivot_credit.div(domain_totals, axis=0).fillna(0)

fig, ax = plt.subplots(figsize=(8.2, 4.8))
x = np.arange(len(domain_order))
bottom = np.zeros(len(domain_order))
for m in model_order:
    vals = pivot_credit[m].to_numpy()
    pct_vals = pct[m].to_numpy()
    ax.bar(x, vals, bottom=bottom, color=colors[m], edgecolor='black', linewidth=0.6, label=m)
    for i, (b, v, p) in enumerate(zip(bottom, vals, pct_vals)):
        if v > 0 and p >= 0.14:
            ax.text(x[i], b + v/2, f'{p*100:.0f}%', ha='center', va='center', fontsize=8, color='white' if m in ['GPT','DeepSeek','Claude'] else 'black')
    bottom += vals
for i, total in enumerate(domain_totals.to_numpy()):
    ax.text(x[i], total + 0.25, str(int(round(total))), ha='center', va='bottom', fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(domain_order, rotation=20, ha='right')
ax.set_ylabel('Number of Novel Strategies in Domain')
ax.set_xlabel('Domain')
ax.set_title('Novel Strategies by Domain and Model Contribution')
ax.legend(frameon=False, ncol=2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## Domain Coverage and Novelty Summary

This table combines two domain-level views of model strategy behavior: (1) how much of the AoPS reference pool is covered by the union of the four models, and (2) how many novel strategies the model union contributes per problem in each domain.

In [ ]:
domain_summary = pd.read_csv(analysis_dir / 'domain_aops_coverage_and_novel_summary_table.csv')
domain_summary_display = domain_summary.copy()
domain_summary_display['AoPS Coverage Rate'] = domain_summary_display['AoPS Coverage Rate'].map(lambda x: f'{x*100:.1f}%')
domain_summary_display['Novel / Problem'] = domain_summary_display['Novel / Problem'].map(lambda x: f'{x:.2f}')
domain_summary_display

## Per-Problem Strategy Inventory

This table shows, for each benchmark problem in the finalized reference set, the number of AoPS-reference strategies (`s#`), the number of novel strategies (`n#`), and the total number of strategies recorded in `complete_strategy_families_AoPS_LLM.csv`.

In [ ]:
per_problem_strategy_inventory = pd.read_csv(analysis_dir / 'per_problem_aops_novel_total_strategy_counts.csv')
per_problem_strategy_inventory

## Three-Run Saturation on 20-Problem Subset

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path('.')
analysis_dir = repo_root / 'analysis_outputs'
overall = pd.read_csv(analysis_dir / 'three_run_same20_overall_summary.csv').iloc[0]

x = ['Full
(same 20)', 'Full +
Run 1', 'Full + Run 1
+ Run 2']
y = [int(overall['full_total']), int(overall['union_full_run1_total']), int(overall['union_all_three_total'])]

fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.plot(range(len(x)), y, marker='o', linewidth=2.5, color='#4E79A7')
ax.fill_between(range(len(x)), y, [min(y)]*len(y), color='#4E79A7', alpha=0.08)
for i, val in enumerate(y):
    ax.text(i, val + 1.8, str(val), ha='center', va='bottom', fontsize=10)
ax.text(0.5, (y[0]+y[1])/2 + 2, '+23', ha='center', va='bottom', fontsize=10, color='#444444')
ax.text(1.5, (y[1]+y[2])/2 + 2, '+10', ha='center', va='bottom', fontsize=10, color='#444444')
ax.set_xticks(range(len(x)))
ax.set_xticklabels(x)
ax.set_ylabel('Distinct strategies discovered')
ax.set_title('Cumulative Strategy Discovery on the 20-Problem Subset')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(min(y)-5, max(y)+12)
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Domain-level delta heatmap: prompt_multi - prompt_single correctness
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

repo_root = Path('.')
analysis_dir = repo_root / 'analysis_outputs'
inp = analysis_dir / 'prompt_single_vs_multi_domain_correctness.csv'
df = pd.read_csv(inp)
models = ["GPT", "Gemini", "DeepSeek", "Claude"]
domains = ["Algebra", "Combinatorics", "Geometry", "Number Theory", "Probability"]

pivot = (
    df.assign(
        domain=pd.Categorical(df['domain'], categories=domains, ordered=True),
        benchmark_model=pd.Categorical(df['benchmark_model'], categories=models, ordered=True),
    )
    .pivot(index='domain', columns='benchmark_model', values='delta_pct')
    .loc[domains, models]
)

fig, ax = plt.subplots(figsize=(7.2, 5.2))
im = ax.imshow(pivot.values, cmap='RdBu_r', vmin=-25, vmax=25, aspect='auto')

ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, fontsize=12)
ax.set_yticks(range(len(domains)))
ax.set_yticklabels(domains, fontsize=12)
ax.set_xlabel('Model', fontsize=13)
ax.set_ylabel('Domain', fontsize=13)
ax.set_title('Domain-level change in correctness under prompt_multi', fontsize=14, pad=12)
ax.tick_params(axis='x', labelrotation=0, length=0)
ax.tick_params(axis='y', labelrotation=0, length=0)

for i in range(len(domains)):
    for j in range(len(models)):
        ax.text(j, i, f"{pivot.values[i, j]:.1f}", ha='center', va='center', fontsize=12, color='black')

cbar = fig.colorbar(im, ax=ax)
cbar.set_label('prompt_multi - prompt_single (pct. points)', size=12)
cbar.ax.tick_params(labelsize=11)

out = repo_root / 'assets' / 'main_text_figures' / 'prompt_single_vs_multi_delta_heatmap_preview.png'
plt.tight_layout()
plt.savefig(out, dpi=220, bbox_inches='tight')
out
